# 02 — Expérimentation des modèles

Le code de production est dans `src/training`. Ce carnet sert à explorer : comparaison des runs MLflow, analyse des erreurs, surapprentissage.

In [ ]:
import os, sys
os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
sys.path.insert(0, os.getcwd())
import pandas as pd
import matplotlib.pyplot as plt
from src.common.config import charger_config
cfg = charger_config()

## 1. Runs MLflow

In [ ]:
import mlflow
from src.common.config import uri_mlflow
mlflow.set_tracking_uri(uri_mlflow(cfg))
runs = mlflow.search_runs(experiment_names=[cfg["mlflow"]["experience"]])
colonnes = ["tags.mlflow.runName", "metrics.val_mae", "metrics.val_wape", "metrics.train_mae", "metrics.ecart_surapprentissage", "metrics.test_mae"]
runs[[c for c in colonnes if c in runs]].sort_values("metrics.val_mae")

## 2. Erreurs du modèle retenu sur le jeu de test

In [ ]:
from src.features.construction import FEATURES, CIBLE
from src.training.registre import charger_champion
from src.evaluation.evaluation import metriques
modele, version = charger_champion(cfg)
test = pd.read_parquet("data/features/features.parquet").query("jeu == 'test'").copy()
test["prediction"] = modele.predict(test[FEATURES]).clip(min=0)
print("Version", version, metriques(test[CIBLE], test["prediction"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
test.assign(e=(test["prediction"] - test[CIBLE]).abs()).groupby("heure_jour")["e"].mean().plot.bar(ax=axes[0], title="MAE par heure")
test.assign(e=test["prediction"] - test[CIBLE]).groupby("station")["e"].mean().plot.barh(ax=axes[1], title="Biais par station")
axes[2].scatter(test[CIBLE], test["prediction"], s=3, alpha=0.3); axes[2].plot([0, test[CIBLE].max()], [0, test[CIBLE].max()], "r--"); axes[2].set_title("Prédit contre réel")
plt.tight_layout()

## 3. Importance des variables (LightGBM)

In [ ]:
if hasattr(modele, "feature_importances_"):
    pd.Series(modele.feature_importances_, index=FEATURES).sort_values().plot.barh(title="Importance des variables")

## 4. Conclusions

*À rédiger :* modèle retenu et pourquoi, heures et stations difficiles, limites.